# 02 · Free-Text Notes

**Goal:** turn free-text time notes into the shared entry table that matching takes.

Many time notes have no columns: a sticky note, a napkin, a notepad page, a typed note, a text message, an email. All of them are free text, so they share one converter. The text goes through the text parser (`src/text_parser.py`, first built for emails in matching notebook 00), which finds hours, CMIDs, clients, and matters in plain text. Each entry it finds becomes one row of the shared table, with a timekeeper, a work date, and a flag when something needs a person to check it.

Matching only knows four source formats (typed, PDF, handwritten, email), so a photographed note is labeled handwritten, and typed text is labeled email since it's parsed the same way.

| Input | How the text is read | Timekeeper | Work date |
|---|---|---|---|
| Photographed note | Textract, line by line (notebook 00) | whoever uploads it | a date written on the note, or the upload date |
| Typed note or text | as typed | whoever uploads it | same |
| Email | the email body | the sender, looked up by email or name | a date in the body, or the upload date |

**Questions**
1. Does the parser handle the way people jot down time on loose notes?
2. Do entries get the right timekeeper and work date?
3. Which entries get flagged for a check, and why?
4. How do photographed notes compare with typed ones?

## 1. Setup

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, "../src")
sys.path.insert(0, "../../matching/src")   # the firm's client and matter name tables
from extract import extract
from fuzzy import client_name_table, matter_name_table
from grade import digits, similar
from intake import CHECK_REASONS, add_day_check, find_timekeeper, timekeeper_table
from sheets import has_table
from textract import load_pages

In [2]:
DATA_PATH = Path("../../matching/data/synthetic_time_entries_v3.xlsx")
MATCHING_CONFIG = Path("../../matching/results/02_config.json")   # billing shorthand never read as client initials
TIMEKEEPERS_PATH = Path("../data/timekeepers.csv")
NOTES_DIR = Path("../data/notes")
TEXTRACT_DIR = Path("../data/textract")
NOTES_KEY_PATH = Path("../data/notes_answer_key.csv")

UPLOAD_DATE = "2026-11-02"   # stands in for the day a note is uploaded
UPLOADER = "TK102"           # stands in for the logged-in timekeeper (R. Patel)
GRADED_FIELDS = ["client", "hours", "cmid", "work_date"]

## 2. Load data

The parser looks names up in the firm's client and matter lists. In the app these come from matching's `load_reference`; here they are built the same way from the synthetic Matter Master.

In [3]:
matter_master = pd.read_excel(DATA_PATH, sheet_name="Matter Master", dtype=str)
acronym_skip = json.loads(MATCHING_CONFIG.read_text())["acronym_skip"]
ref = {"client_names": client_name_table(matter_master, acronym_skip), "matter_names": matter_name_table(matter_master)}
timekeepers = timekeeper_table(pd.read_csv(TIMEKEEPERS_PATH))

assert timekeepers["tk_id"].is_unique and timekeepers["email"].is_unique
assert all(find_timekeeper(n, timekeepers) for n in ["R. Patel", "E. Brandt", "L. Moreno", "A. Chen"])
print(f"{matter_master['Client'].nunique()} clients | {len(ref['client_names'])} client names, aliases, and initials | "
      f"{len(timekeepers)} timekeepers")

35 clients | 83 client names, aliases, and initials | 18 timekeepers


## 3. Typed notes

Short notes written the way people jot time down, using the synthetic firm's clients. Each one lists what a person would pull out of it. Notes with no date written on them should get the upload date.

In [4]:
examples = [
    {"layout": "Shorthand line", "text": ".4 Halvorsen - call w/ CEO re hiring",
     "expected": [{"client": "Halvorsen", "hours": 0.4, "cmid": None}]},
    {"layout": "Date on top", "text": "10/13\nCobalt Ridge Partners\nrevise sub docs for Fund II 1.2",
     "expected": [{"client": "Cobalt Ridge Partners", "hours": 1.2, "cmid": None, "work_date": "2026-10-13"}]},
    {"layout": "Two entries", "text": "Meridian - proxy draft 1.5\nTidewater - board consent .3",
     "expected": [{"client": "Meridian", "hours": 1.5, "cmid": None}, {"client": "Tidewater", "hours": 0.3, "cmid": None}]},
    {"layout": "CMID instead of client", "text": "5520.001 .6 review purchase agmt",
     "expected": [{"client": None, "hours": 0.6, "cmid": "5520.001"}]},
    {"layout": "Minutes", "text": "45 min Bluefin - option grant call",
     "expected": [{"client": "Bluefin", "hours": 0.75, "cmid": None}]},
    {"layout": "Hours and minutes", "text": "Kestrel Foods 1:15 bakery APA markup",
     "expected": [{"client": "Kestrel Foods", "hours": 1.25, "cmid": None}]},
    {"layout": "Client typo", "text": "Halvorson ind - TCW lender counsel re Brightline 2.1",
     "expected": [{"client": "Halvorson ind", "hours": 2.1, "cmid": None}]},
    {"layout": "Initials as a header", "text": "HLP\n- LPA comments .8\n- side letter tracker .4",
     "expected": [{"client": "HLP", "hours": 0.8, "cmid": None}, {"client": "HLP", "hours": 0.4, "cmid": None}]},
    {"layout": "Day header", "text": "Mon 10/13\n.3 Ironwood 10-Q comments\n1.1 Ironwood - Project Prism diligence calls",
     "expected": [{"client": "Ironwood", "hours": 0.3, "cmid": None, "work_date": "2026-10-13"},
                  {"client": "Ironwood", "hours": 1.1, "cmid": None, "work_date": "2026-10-13"}]},
    {"layout": "Same as above", "text": "Meridian proxy .5\nsame - call w/ GC re timing .3",
     "expected": [{"client": "Meridian", "hours": 0.5, "cmid": None}, {"client": "Meridian", "hours": 0.3, "cmid": None}]},
    {"layout": "Firm time", "text": "firm admin - update time entries .2",
     "expected": [{"client": None, "hours": 0.2, "cmid": None}]},
    {"layout": "No hours", "text": "Saffron Labs - review seed docs",
     "expected": [{"client": "Saffron Labs", "hours": None, "cmid": None}]},
]
for ex in examples:
    ex["expected"] = [{"work_date": UPLOAD_DATE, **e} for e in ex["expected"]]
print(examples[1]["text"])

10/13
Cobalt Ridge Partners
revise sub docs for Fund II 1.2


In [5]:
def as_found(entries):
    found = entries[GRADED_FIELDS].assign(work_date=entries["work_date"].dt.strftime("%Y-%m-%d"))
    return found.astype(object).where(found.notna(), None).to_dict("records")


notes, graded = [], []
for i, ex in enumerate(examples, 1):
    entries = extract(ex["text"], "TEXT", ref, timekeepers, tk_id=UPLOADER, upload_date=UPLOAD_DATE, source=f"note_{i:02d}")
    notes.append(entries.assign(layout=ex["layout"]))
    graded.append({"layout": ex["layout"], "entries": len(entries), "as_expected": as_found(entries) == ex["expected"]})
notes = pd.concat(notes, ignore_index=True)
graded = pd.DataFrame(graded)
print(f"{graded['as_expected'].sum()} of {len(graded)} notes parsed as expected")
graded

12 of 12 notes parsed as expected


,layout,entries,as_expected
0,Shorthand line,1,True
1,Date on top,1,True
2,Two entries,2,True
3,CMID instead of client,1,True
4,Minutes,1,True
5,Hours and minutes,1,True
6,Client typo,1,True
7,Initials as a header,2,True
8,Day header,2,True
9,Same as above,2,True


In [6]:
notes[["layout", "entry_id", "tk_id", "work_date", "cmid", "client", "matter", "narrative", "hours", "check_reasons"]]

,layout,entry_id,tk_id,work_date,cmid,client,matter,narrative,hours,check_reasons
0,Shorthand line,note_01,TK102,2026-11-02,None,Halvorsen,None,call w/ CEO re hiring,0.40,
1,Date on top,note_02,TK102,2026-10-13,None,Cobalt Ridge Partners,None,revise sub docs for Fund II,1.20,
2,Two entries,note_03-1,TK102,2026-11-02,None,Meridian,None,proxy draft,1.50,
3,Two entries,note_03-2,TK102,2026-11-02,None,Tidewater,None,board consent,0.30,
4,CMID instead of client,note_04,TK102,2026-11-02,5520.001,None,None,review purchase agmt,0.60,
5,Minutes,note_05,TK102,2026-11-02,None,Bluefin,None,option grant call,0.75,
6,Hours and minutes,note_06,TK102,2026-11-02,None,Kestrel Foods,None,bakery APA markup,1.25,
7,Client typo,note_07,TK102,2026-11-02,None,Halvorson ind,None,TCW lender counsel re Brightline,2.10,
8,Initials as a header,note_08-1,TK102,2026-11-02,None,HLP,None,LPA comments,0.80,
9,Initials as a header,note_08-2,TK102,2026-11-02,None,HLP,None,side letter tracker,0.40,


- All 12 note styles came out as expected: hours as decimals, minutes, and clock style, CMIDs, initials, a client typo, and a client header over several lines.
- A date written on the note becomes the work date. Notes without one get the upload date.
- "same" copies the client from the line above and flags it, so the reviewer sees it was copied.
- Firm time ("firm admin") leaves the client blank on purpose. "Firm" is an ordinary word, so matching reads firm time from the narrative instead.
- These notes were written to test the rules, so a perfect score here only shows the rules work. Photographed notes are the real test.

## 4. Emails

For an email, the timekeeper is the sender. The sender's address is looked up in the timekeeper list's Email column, and a display name with no known address falls back to a name lookup. A sender who isn't a timekeeper (an assistant forwarding time, for example) is flagged.

In [7]:
emails = [
    {"sender": "rpatel@example.com", "text": "Hi Pat,\n\nPlease enter: Ridgewater Fund III - LPA comments 1.4\n\nThx\nR"},
    {"sender": "Elena Brandt <elena.b@gmail.com>", "text": "Tidewater AI (4590.003) - charter amendment for Project Kite .9\n\nSent from my iPhone"},
    {"sender": "pat.lee@example.com", "text": "Fwd from Alex: Harborlight Bio / Series A Financing - board consent .5"},
]
email_entries = pd.concat([
    extract(e["text"], "EMAIL", ref, timekeepers, tk_name=e["sender"], upload_date=UPLOAD_DATE, source=f"email_{i}")
    for i, e in enumerate(emails, 1)], ignore_index=True)
email_entries[["entry_id", "tk_name", "tk_id", "work_date", "cmid", "client", "matter", "narrative", "hours", "check_reasons"]]

,entry_id,tk_name,tk_id,work_date,cmid,client,matter,narrative,hours,check_reasons
0,email_1,rpatel@example.com,TK102,2026-11-02,None,Ridgewater,None,Fund III - LPA comments,1.4,
1,email_2,Elena Brandt <elena.b@gmail.com>,TK109,2026-11-02,4590.003,Tidewater AI,None,charter amendment for Project Kite,0.9,
2,email_3,pat.lee@example.com,None,2026-11-02,None,Harborlight Bio,Series A Financing,Fwd from Alex: - board consent,0.5,unknown_tk


- The sender's address finds the timekeeper, and a personal address falls back to the display name (Elena Brandt to E. Brandt).
- An email from someone who isn't a timekeeper is flagged rather than guessed. For a forward, the real timekeeper (Alex) is named in the text, so a reviewer confirms it.
- Forwarding text ("Fwd from Alex:") stays in the narrative, which a reviewer can trim.

## 5. Checks across a day

A timekeeper rarely bills more than about 14 hours in a day, so a day over that usually means a misread hour. The check runs on the whole table, across uploads, so two notes from the same day are added together.

In [8]:
morning = extract("Ridgewater Fund IV kickoff 9.0", "TEXT", ref, timekeepers, tk_id=UPLOADER,
                  upload_date=UPLOAD_DATE, source="note_am")
evening = extract("Ridgewater - LPA turn 6.5", "TEXT", ref, timekeepers, tk_id=UPLOADER,
                  upload_date=UPLOAD_DATE, source="note_pm")
day = add_day_check(pd.concat([morning, evening], ignore_index=True))
day[["entry_id", "tk_id", "work_date", "client", "hours", "check_reasons"]]

,entry_id,tk_id,work_date,client,hours,check_reasons
0,note_am,TK102,2026-11-02,Ridgewater,9.0,day_over_max
1,note_pm,TK102,2026-11-02,Ridgewater,6.5,day_over_max


In [9]:
pd.Series(CHECK_REASONS, name="shown to the reviewer").to_frame()

,shown to the reviewer
unknown_tk,Timekeeper not found
no_hours,No hours found
no_date,No work date found
day_over_max,Over 14 hours on one day
low_conf,Hard to read
hours_10x,Hours may be missing a decimal
client_copied,Client copied from the line above
date_guessed,"Date taken from the week, not the row"


- Each upload is fine on its own (9.0 and 6.5 hours), but together they come to 15.5 hours on one day, so both are flagged.
- In the app, this check runs on everything a timekeeper has entered for the day, not one upload at a time.

## 6. Photographed notes

Photos go in `data/notes/` and are read by Textract in notebook 00. This section picks up every photo with a saved Textract answer and runs it through the same converter. A photo where Textract found columns drawn on the page is read as a table instead, in notebook 01, since reading it line by line splits each entry across several lines. The rest are split into entries line by line: a line with hours starts an entry, lines below it join it, and a line that's only a date applies to the entries under it. A few common photo misreads are fixed first ("05" for .5, "10127" for 10/27), and hours fixed this way are flagged. Each photo is graded against `data/notes_answer_key.csv` (one row per entry, in the order written). The client only has to be similar to what is written, since misread names are left for matching, and CMIDs are compared by their digits. The narrative only has to be roughly the same (70 or more out of 100), which catches text that went missing without counting OCR typos.

In [10]:
read = sorted(p for p in NOTES_DIR.glob("*") if load_pages(TEXTRACT_DIR, p, "note"))
photos = [p for p in read if not has_table(load_pages(TEXTRACT_DIR, p, "sheet"))]
print(f"{len(read) - len(photos)} photos with drawn columns are read as tables in notebook 01")
if photos:
    photo_entries = pd.concat([
        extract(load_pages(TEXTRACT_DIR, p, "note"), "NOTE", ref, timekeepers, tk_id=UPLOADER,
                upload_date=UPLOAD_DATE, source=p.name)
        for p in photos], ignore_index=True)
    display(photo_entries[["entry_id", "work_date", "cmid", "client", "narrative", "hours", "min_conf", "check_reasons"]])
else:
    print("No free-text note photos read yet. Add photos to data/notes/ and run notebook 00.")

2 photos with drawn columns are read as tables in notebook 01


,entry_id,work_date,cmid,client,narrative,hours,min_conf,check_reasons
0,note_01,2026-11-02,None,Halvorsen,4 - call w/lender counselve Brightline closing,NaN,58.948303,low_conf; no_hours
1,note_02,2026-10-27,5602.003,None,review Quanta disclosure schedules,1.60,87.036636,
2,note_03-1,2026-11-02,None,Ironwood,diligence calls w/ Quanta mgmt,2.50,95.986923,
3,note_03-2,2026-11-02,None,Ironwood,markup SPA,0.80,39.085022,low_conf; hours_10x; client_copied
4,note_04-1,2026-11-02,None,Tidewater Analytics,draft side letter for Series A & arm lead,1.20,23.014191,low_conf
5,note_04-2,2026-11-02,None,NaN,Firm admin 3 time entries,0.30,87.922409,
6,note_05-1,2026-10-26,None,Orchard Lane,review Vendor contract,0.90,74.422234,low_conf
7,note_05-2,2026-10-21,None,Solace Health,board minutes,1.10,64.157166,low_conf
8,note_06-1,2026-11-02,None,Ridgeview,call re capital call,0.50,62.852360,low_conf; hours_10x
9,note_06-2,2026-11-02,None,NaN,Firm BD lunch w/ prospective client,1.00,83.465401,


In [11]:
key = pd.read_csv(NOTES_KEY_PATH, dtype={"cmid": str})
key["work_date"] = pd.to_datetime(key["work_date"].fillna(UPLOAD_DATE))
found = photo_entries.rename(columns={"source": "photo"})
found["entry"] = found.groupby("photo").cumcount() + 1

counts = pd.DataFrame({"expected": key.groupby("photo").size(), "found": found.groupby("photo").size()})
print(f"{(counts['expected'] == counts['found']).sum()} of {len(counts)} photos split into the right number of entries")
counts

7 of 7 photos split into the right number of entries


,expected,found
photo,,
note_01.JPG,1,1
note_02.JPG,1,1
note_03.JPG,2,2
note_04.JPG,2,2
note_05.JPG,2,2
note_06.JPG,2,2
note_08.JPG,1,1


In [12]:
graded = key.merge(found, on=["photo", "entry"], how="left", suffixes=("_key", ""))
right = pd.DataFrame({
    "hours": graded["hours"].round(2) == graded["hours_key"].round(2),
    "client": [similar(f, t) for f, t in zip(graded["client"], graded["client_key"])],
    "narrative": [similar(f, t, 70) for f, t in zip(graded["narrative"], graded["narrative_key"])],
    "cmid": digits(graded["cmid"]) == digits(graded["cmid_key"]),
    "date": graded["work_date"] == graded["work_date_key"],
})
print(right.mean().round(2).to_string())
graded.join(right.add_suffix("_right"))[["photo", "entry", "hours", "hours_key", "client", "client_key", "cmid",
                                         "work_date", "work_date_key", "narrative", "narrative_key", "check_reasons"]]

hours        0.91
client       0.91
narrative    0.91
cmid         1.00
date         0.91


,photo,entry,hours,hours_key,client,client_key,cmid,work_date,work_date_key,narrative,narrative_key,check_reasons
0,note_01.JPG,1,NaN,0.40,Halvorsen,Halvorsen,None,2026-11-02,2026-11-02,4 - call w/lender counselve Brightline closing,call w/lender counsel re Brightline closing,low_conf; no_hours
1,note_02.JPG,1,1.60,1.60,None,NaN,5602.003,2026-10-27,2026-10-27,review Quanta disclosure schedules,review Quanta disclosure schedules,
2,note_03.JPG,1,2.50,2.50,Ironwood,Ironwood,None,2026-11-02,2026-11-02,diligence calls w/ Quanta mgmt,diligence calls w/ Quanta mgmt,
3,note_03.JPG,2,0.80,0.80,Ironwood,Ironwood,None,2026-11-02,2026-11-02,markup SPA,markup SPA,low_conf; hours_10x; client_copied
4,note_04.JPG,1,1.20,1.20,Tidewater Analytics,Tidewater Analytics,None,2026-11-02,2026-11-02,draft side letter for Series A & arm lead,draft side letter for Series A lead,low_conf
5,note_04.JPG,2,0.30,0.30,NaN,NaN,None,2026-11-02,2026-11-02,Firm admin 3 time entries,Firm admin time entries,
6,note_05.JPG,1,0.90,0.90,Orchard Lane,Orchard Lane,None,2026-10-26,2026-10-26,review Vendor contract,review vendor contract,low_conf
7,note_05.JPG,2,1.10,1.10,Solace Health,Solace Health,None,2026-10-21,2026-10-27,board minutes,board minutes,low_conf
8,note_06.JPG,1,0.50,0.50,Ridgeview,Ridgeview,None,2026-11-02,2026-11-02,call re capital call,call re capital call,low_conf; hours_10x
9,note_06.JPG,2,1.00,1.00,NaN,NaN,None,2026-11-02,2026-11-02,Firm BD lunch w/ prospective client,Firm BD lunch w/ prospective client,


- All 7 photos split into the right number of entries, including entries that wrap onto a second line, two dates on one note, and a "same" line.
- CMIDs are right on every entry, and hours, clients, narratives, and dates on 10 of 11.
- Each miss comes from Textract misreading a small mark: ".4" read as "4" (note 1, flagged as missing hours), "10/27" read as "10/21" (note 5), and "Ironwood" read as "tronwood" next to a scribble read as "$77" (note 8).
- Two decimals were recovered by the photo fixes (".5" read as "05", ".8" read as ", 8"), and both are flagged so a reviewer confirms them.
- The wrong date on note 5 can't be caught, since "10/21" is a real date.
- 7 of 11 entries are flagged as hard to read, so photographed notes lean on review more than typed ones.

## 7. Summary

Typed notes and photographed notes side by side. A file is split right when it gives the right number of entries. The field columns are the share of entries with that field right (narratives aren't graded for typed notes, since they're typed exactly).

In [13]:
typed = []
for ex in examples:
    found_typed = as_found(notes[notes["layout"] == ex["layout"]])
    typed += [{field: f[field] == e[field] for field in GRADED_FIELDS} for f, e in zip(found_typed, ex["expected"])]
typed = pd.DataFrame(typed).rename(columns={"work_date": "date"})
typed_split = sum(len(notes[notes["layout"] == ex["layout"]]) == len(ex["expected"]) for ex in examples)

summary = pd.DataFrame({
    "Typed notes": {"files right": typed_split, "files": len(examples), "entries": len(notes), **typed.mean()},
    "Photos": {"files right": (counts["expected"] == counts["found"]).sum(), "files": len(counts), "entries": len(found),
               **right.mean()},
}).T[["files right", "files", "entries", "hours", "client", "cmid", "date", "narrative"]]
rates = ["hours", "client", "cmid", "date", "narrative"]
summary[rates] = summary[rates].astype(float).round(2)
summary

,files right,files,entries,hours,client,cmid,date,narrative
Typed notes,12.0,12.0,16.0,1.00,1.00,1.0,1.00,NaN
Photos,7.0,7.0,11.0,0.91,0.91,1.0,0.91,0.91


- Every typed note and every photo splits into the right number of entries.
- Typed notes get every field right. Photos get the CMID right every time and the other fields on 10 of 11 entries.
- The gap between the two is the cost of reading a photo: each miss traces back to a small mark Textract misread, not to the free-text rules.

## Findings

### Results
- **Typed notes:** 12 of 12 parsed as expected across 12 common styles.
- **Emails:** the timekeeper comes from the sender, and unknown senders are flagged.
- **Checks:** missing hours, a copied client, an unknown timekeeper, and a day over 14 hours all go to review instead of being guessed.
- **Photographed notes:** all 7 split into the right number of entries, with hours, client, date, and narrative right on 10 of 11. Every miss traces back to a misread mark.

### Tradeoffs
- One converter for every kind of free text keeps the rules in one place: a fix for emails also fixes sticky notes and text messages.
- Notes are split line by line (an entry starts at its hours, and the lines below join it), which handles wrapped lines and several dates. A client name written alone on a line between entries is read as a header for the entries below it.
- The text parser was first built for emails, so a note format it doesn't know (hours written as "point four", arrows) needs a new rule rather than a new model.

### Limitations
- The typed notes were written to test the parser, so they are cleaner and more predictable than real notes.
- The photo fixes ("05" read as .5) are guesses, so they're flagged, and a decimal point that disappears completely can't be recovered.
- A date that looks like a fraction ("2/3") can be read as a date.

### Next steps
- Photograph 8 to 10 real notes (different surfaces, neat and messy writing, one and several entries), fill in the answer key, and grade them here.
- Run the extracted entries through matching's `decide()` (notebook 03).